# `Plot` — Courbes Cartésiennes 2D (`fx.Plot` / `fig.plot()`)

Hiérarchie OOP : `Visual` → `Axis2D` → **`Plot`**.
Survolez les courbes pour inspecter les valeurs exactes via la **mire magnétique multi-séries**, zoomez à la molette sur un détail et glissez pour naviguer.

## 1. Exemple minimal (1 ligne)
Passer une seule séquence `y` génère automatiquement `x = [0, 1, ..., N-1]` :

In [1]:
import fluxio as fx

fx.Plot([12, 28, 19, 37, 31, 46], label="Requêtes", size=(760, 340))

## 2. Superposition multi-séries, styles de traits (`solid`, `dashed`, `dotted`), marqueurs & aire dégradée
Appeler `.plot(...)` plusieurs fois sur le même objet (ou la même case `fig[r, c]`) superpose les courbes avec légende intégrée :

In [2]:
import math

xs = [round(i * 0.2, 2) for i in range(40)]
y_model = [80.0 * math.exp(-0.24 * x) + 20.0 for x in xs]
y_sensor = [80.0 * math.exp(-0.24 * x) + 20.0 + 8.5 * math.sin(x * 2.5) for x in xs]
y_target = [20.0 for _ in xs]

p = fx.Plot(
    xs,
    y_model,
    label="Modèle théorique",
    xlabel="Temps (s)",
    ylabel="Température (°C)",
    title="Refroidissement Thermique Multi-Capteurs",
    subtitle="Survolez le graphique pour lire simultanément les 3 séries",
    fill_alpha=0.16,
    size=(880, 420),
)
p.style(
    background=fx.Gradient(["#090d16", "#0f172a", "#1e1b4b"], angle=135),
    accent="#38bdf8",
    accent_side="left",
    radius=20,
)
p.plot(
    xs,
    y_sensor,
    label="Capteur IR",
    color="#f97316",
    linestyle="dashed",
    marker="circle",
    fill_alpha=0.04,
)
p.plot(
    xs,
    y_target,
    label="Ambiante (20 °C)",
    color="#34d399",
    linestyle="dotted",
    linewidth=1.8,
    fill=False,
)
p

<fluxio.Plot 'Refroidissement Thermique Multi-Capteurs'>

## 2b. Contrôle de la grille et des bornes (`xmin`/`xmax`, `ymin`/`ymax`, `xstep`/`ystep`)
- Fixez les deux bornes ou **une seule borne** (ex. `ymin=0` verrouille le plancher à `0` tout en laissant `ymax` s'adapter automatiquement).
- Imposez le pas exact des colonnes/lignes avec `xstep=2`, `ystep=20` (ou le raccourci tuple `xlim=(0, 10, 2)`) :

In [3]:
gx = [round(i * 0.25, 2) for i in range(41)]  # 0 .. 10

fig_grid = fx.Figure(
    rows=1,
    cols=2,
    title="Contrôle Précis des Bornes & du Pas de la Grille",
    size=(940, 370),
)
fig_grid.style(background=fx.Gradient(["#090d16", "#0f172a"], angle=135), radius=18)

# Gauche : xmin=0, xmax=10, xstep=2 -> colonnes à 0, 2, 4, 6, 8, 10 ; ymin=0, ymax=100, ystep=20
fig_grid[0, 0].style(accent="#38bdf8", accent_side="left").plot(
    gx,
    [50.0 + 38.0 * math.sin(x * 0.8) for x in gx],
    label="Signal A",
    card_title="Grille fixe (xstep=2, ystep=20)",
    xlabel="Distance (m)",
    ylabel="Amplitude",
    xmin=0,
    xmax=10,
    xstep=2,
    ymin=0,
    ymax=100,
    ystep=20,
)

# Droite : raccourci xlim=(0, 10, 1) + borne partielle ymin=0 + légende en bas
fig_grid[0, 1].style(accent="#a78bfa", accent_side="right").plot(
    gx,
    [25.0 + 48.0 * (math.sin(x * 0.55) ** 2) for x in gx],
    label="Puissance (W)",
    card_title="xlim=(0, 10, 1) + ymin=0 (plafond auto)",
    xlabel="Temps (s)",
    xlim=(0, 10, 1),
    ymin=0,
    ystep=15,
    legend="bottom",
)
fig_grid

<fluxio.Figure 'Contrôle Précis des Bornes & du Pas de la Grille' grid=1x2 cells=2>

## 3. Les 3 variantes visuelles (`classic`, `reverse`, `text`) sur `Plot`

In [4]:
w_x = [round(i * 0.25, 2) for i in range(25)]
w_y = [math.sin(x) * 40 + 50 for x in w_x]

fig_var = fx.Figure(
    rows=1,
    cols=3,
    title="Variantes de Cartes sur Plot",
    size=(960, 360),
)
fig_var.style(background=fx.Gradient(["#090d16", "#172554"], angle=125), radius=18)

fig_var[0, 0].style(variant="classic", accent="#38bdf8", accent_side="left").plot(
    w_x, w_y, label="Classic", card_title="Classic", xlabel="t (s)"
)
fig_var[0, 1].style(variant="reverse", accent="#059669", accent_side="top").plot(
    w_x, w_y, label="Reverse", card_title="Reverse", xlabel="t (s)"
)
fig_var[0, 2].style(variant="text", accent="#f97316", accent_side="right").plot(
    w_x, w_y, label="Text TUI", card_title="Text TUI", xlabel="t (s)"
)
fig_var

<fluxio.Figure 'Variantes de Cartes sur Plot' grid=1x3 cells=3>

## 4. Dynamique Temps Réel : Onde progressive `f(t) -> (xs, ys)` vs Fenêtre glissante (`yield`)
- Retourner un couple `(xs, ys)` anime toute la courbe à chaque instant `t`.
- Retourner un scalaire `float` (via `f(t)` ou un générateur `yield`) active automatiquement une **fenêtre glissante** de taille `max_points` :

In [5]:
import itertools

wave_xs = [round(i * 0.16, 2) for i in range(44)]

live_plot = fx.Figure(
    rows=1,
    cols=2,
    title="Animations Temps Réel — Onde Progressive & Télémétrie Glissante",
    size=(940, 380),
    fps=24,
)
live_plot.style(background=fx.Gradient(["#090d16", "#0f172a", "#1e1b4b"], angle=135), radius=20)

# Gauche : Onde animée complète f(t) -> (xs, ys)
live_plot[0, 0].style(accent="#38bdf8", accent_side="left").plot(
    lambda t: (
        wave_xs,
        [math.sin(x - t * 3.0) * math.cos(x * 0.25) for x in wave_xs],
    ),
    label="Onde Primaire",
    card_title="Mode Onde — f(t) -> (xs, ys)",
    xlabel="Position (m)",
    ylim=(-1.15, 1.15),
)
live_plot[0, 0].plot(
    lambda t: (
        wave_xs,
        [0.5 * math.cos(x * 1.2 + t * 3.0) for x in wave_xs],
    ),
    label="Écho",
    color="#f97316",
    linestyle="dashed",
    fill_alpha=0.04,
)

# Droite : Fenêtre glissante alimentée par un générateur Python
telemetry_gen = (
    50.0 + 28.0 * math.sin(i * 0.24) + 9.0 * math.cos(i * 0.65)
    for i in itertools.count()
)
live_plot[0, 1].style(accent="#34d399", accent_side="right").plot(
    telemetry_gen,
    label="Charge CPU (%)",
    card_title="Mode Fenêtre Glissante — yield",
    xlabel="Échantillon",
    ylim=(0.0, 100.0),
    max_points=45,
)
live_plot

<fluxio.Figure 'Animations Temps Réel — Onde Progressive & Télémétrie Glissante' grid=1x2 cells=2>

## 5. Dashboard Hybride (`Figure` avec `col_span` : `Number` + `Plot`)

In [6]:
dash = fx.Figure(
    rows=2,
    cols=2,
    title="Supervision Cluster Inférence IA",
    size=(920, 520),
)
dash.style(background=fx.Gradient(["#090d16", "#0f172a"], angle=140), radius=18, gap=16)

dash[0, 0].style(accent="#38bdf8", accent_side="left").number(
    1420, unit="tok/s", precision=0, label="Débit Global", description="8x GPU actifs"
)
dash[0, 1].style(variant="reverse", accent="#059669", accent_side="right").number(
    24.6, unit="ms", precision=1, label="TTFT Médian", description="Time-To-First-Token"
)

hours = list(range(24))
dash[1, 0].style(accent="#38bdf8", accent_side="bottom").plot(
    hours,
    [620 + 580 * math.sin((h - 6) * math.pi / 12) ** 2 for h in hours],
    label="Débit (tok/s)",
    card_title="Historique sur 24h (col_span=2)",
    xlabel="Heure (h)",
    ylabel="tok/s",
    marker="dot",
    col_span=2,
)
dash

<fluxio.Figure 'Supervision Cluster Inférence IA' grid=2x2 cells=3>

## 6. Rendu Terminal Haute Résolution (Braille Unicode `2×4`)

In [7]:
bx = [i * 0.25 for i in range(28)]
bp = fx.Plot(bx, [math.sin(x) for x in bx], label="sin(x)", xlabel="x")
bp.plot(bx, [math.cos(x) for x in bx], label="cos(x)", linestyle="dashed", color="#f97316")
print(bp.to_text(ansi=False, width=74))

╭━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╮
│ ━ sin(x)  ╌ cos(x)                                                     │
│  1.12┤⠤⠤⠠⢄⣀⡀    ⢀⡠⠤⠤⠤⠤⠤⠤⣀⡀                                   ⣀⠠⠤⠄⠤⠤⠤⢄  │
│      │     ⠈⣁⠦⢖⠉⠁        ⠈⠑⠒⠤⣀                          ⣀⠤⠔⠂⠉          │
│      │   ⡠⠔⠉   ⠈⠒⠄⡀           ⠉⠒⢄                    ⢀⠤⠈            ⣀⠔ │
│      │⢀⠤⠊         ⠈⠢⡀            ⠑⢄⡀              ⢀⠤⠂⠁           ⢀⠤⠊   │
│     0┤⠁             ⠈⠑⠠⣀           ⠈⠑⠤⣀         ⠠⠊⠁           ⢀⠤⠒⠁     │
│      │                  ⠁⠢⣀            ⠑⠢⢄   ⡀⠔⠉           ⢀⡠⠒⠁        │
│      │                     ⠑⠢⢀⣀         ⢀⢀⠵⠲⢍⡀         ⣀⠤⠒⠊⠁           │
│ -1.12┤                         ⠁⠉⠒⠒⠒⠐⠒⠁⠉⠁    ⠈⠉⠑⠒⠒⠒⠒⠒⠒⠉                │
│      └──────────────────────────────────────────────────────────────── │
│       0                             x                             6.75 │
╰────────────────────────────────────────────────────────────────────────╯
